# 🚗 Notebook 05 — Advanced EV Business Analytics & Feature Engineering

## Overview

This notebook is the **bridge between statistical analysis and dashboard / machine-learning preparation**.

The cleaned and renamed dataset from Notebook 01 is used as the starting point. EDA and statistical testing were already handled in earlier notebooks, so this notebook focuses on:

- Advanced business aggregations
- Market-share analysis
- Manufacturer intelligence
- Model intelligence
- Geographic intelligence
- Vehicle-age features
- Range segmentation
- MSRP segmentation
- CAFV analytical features
- Ranking and percentile analysis
- Portfolio analysis
- Business scoring
- Market segmentation
- ML-ready feature creation
- Exporting reusable analytical datasets

### Input

```text
../datas/cleaned/ev_cleaned.csv
```

### Output

```text
../datas/processed/
├── ev_features.csv
├── manufacturer_analytics.csv
├── model_analytics.csv
├── state_analytics.csv
├── yearly_analytics.csv
├── ev_type_analytics.csv
├── market_share_analytics.csv
└── executive_kpis.csv
```

> **Important:** This notebook does not intentionally recreate the cleaning workflow from Notebook 01.


# 1. Business Questions

The analysis should answer questions such as:

1. Which manufacturers dominate the dataset?
2. Which models have the largest vehicle population?
3. Which states have the strongest EV presence?
4. Which manufacturers have the broadest product portfolios?
5. Which manufacturers have the highest median electric range?
6. How are vehicles distributed across range segments?
7. How are vehicles distributed across MSRP segments?
8. Which vehicle types dominate?
9. Which manufacturers have broad geographic coverage?
10. Which model-year groups contain the largest vehicle population?
11. Which manufacturers combine high volume with high range?
12. Which states show high manufacturer/model diversity?
13. Which vehicle records belong to high/medium/low analytical segments?
14. Which engineered features can be reused in ML and Power BI?


# 2. Load the Cleaned Dataset

The project uses the `datas` directory established in the previous workflow.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_PATH = Path("../datas/cleaned/ev_cleaned.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Cleaned dataset not found: {DATA_PATH.resolve()}"
    )

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())


Shape: (150482, 17)


C:\Users\Admin\AppData\Local\Temp\ipykernel_6696\2478218131.py:15: DtypeWarning: Columns (0: Postal_Code, 1: Census_Tract) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH)


,VIN,County,City,State,Postal_Code,Model_Year,Make,Model,EV_Type,CAFV_Eligibility,Electric_Range,Base_MSRP,Vehicle_ID,Vehicle_Location,Electric_Utility,Census_Tract,Legislative_District
0,KM8K33AGXL,King,Seattle,WA,"98,103.00",2020,HYUNDAI,KONA,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,258,0,249675142,POINT (-122.34301 47.659185),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),"53,033,004,800.00",43
1,1C4RJYB61N,King,Bothell,WA,"98,011.00",2022,JEEP,GRAND CHEROKEE,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,25,0,233928502,POINT (-122.20578 47.762405),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),"53,033,021,804.00",1
2,1C4RJYD61P,Yakima,Yakima,WA,"98,908.00",2023,JEEP,GRAND CHEROKEE,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,25,0,229675939,POINT (-120.6027202 46.5965625),PACIFICORP,"53,077,002,900.00",14
3,5YJ3E1EA7J,King,Kirkland,WA,"98,034.00",2018,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,215,0,104714466,POINT (-122.209285 47.71124),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),"53,033,021,903.00",45
4,WBY7Z8C5XJ,Thurston,Olympia,WA,"98,501.00",2018,BMW,I3,Plug-in Hybrid Electric Vehicle (PHEV),Clean Alternative Fuel Vehicle Eligible,97,0,185498386,POINT (-122.89692 47.043535),PUGET SOUND ENERGY INC,"53,067,010,700.00",22


# 3. Validate the Analytical Starting Point

Check that the cleaned dataset is suitable for advanced analysis.


In [2]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nNumeric columns:")
display(df.select_dtypes(include=np.number).columns.tolist())

print("\nCategorical columns:")
display(df.select_dtypes(exclude=np.number).columns.tolist())


Rows: 150482
Columns: 17

Missing values:


VIN                     0
County                  0
City                    0
State                   0
Postal_Code             0
Model_Year              0
Make                    0
Model                   0
EV_Type                 0
CAFV_Eligibility        0
Electric_Range          0
Base_MSRP               0
Vehicle_ID              0
Vehicle_Location        0
Electric_Utility        0
Census_Tract            0
Legislative_District    0
dtype: int64


Duplicate rows: 0

Numeric columns:


['Model_Year',
 'Electric_Range',
 'Base_MSRP',
 'Vehicle_ID',
 'Legislative_District']


Categorical columns:


['VIN',
 'County',
 'City',
 'State',
 'Postal_Code',
 'Make',
 'Model',
 'EV_Type',
 'CAFV_Eligibility',
 'Vehicle_Location',
 'Electric_Utility',
 'Census_Tract']

# 4. Feature Engineering — Vehicle Age

Vehicle age is calculated relative to the latest model year represented in the dataset.

This avoids hard-coding the current calendar year and keeps the analysis reproducible.


In [3]:
REFERENCE_YEAR = int(df["Model_Year"].max())

df["Vehicle_Age"] = (
    REFERENCE_YEAR - df["Model_Year"]
).clip(lower=0)

display(df[["Model_Year", "Vehicle_Age"]].head())
print("Reference year:", REFERENCE_YEAR)


,Model_Year,Vehicle_Age
0,2020,4
1,2022,2
2,2023,1
3,2018,6
4,2018,6


Reference year: 2024


# 5. Vehicle Age Segmentation

Create interpretable age groups for business analysis.


In [4]:
df["Age_Group"] = pd.cut(
    df["Vehicle_Age"],
    bins=[-1, 2, 5, 10, np.inf],
    labels=[
        "0–2 Years",
        "3–5 Years",
        "6–10 Years",
        "10+ Years"
    ]
)

display(df["Age_Group"].value_counts(dropna=False))


Age_Group
0–2 Years     65520
3–5 Years     40695
6–10 Years    37212
10+ Years      7055
Name: count, dtype: int64

# 6. Electric Range Segmentation

Range categories make range analysis easier across manufacturers, models and states.


In [5]:
df["Range_Category"] = pd.cut(
    df["Electric_Range"],
    bins=[-1, 100, 200, 300, 400, np.inf],
    labels=[
        "0–100",
        "101–200",
        "201–300",
        "301–400",
        "400+"
    ]
)

display(df["Range_Category"].value_counts(dropna=False).sort_index())


Range_Category
0–100      113200
101–200      6613
201–300     28010
301–400      2659
400+            0
Name: count, dtype: int64

# 7. MSRP Segmentation

MSRP is converted into business-friendly price bands.

A zero MSRP should not automatically be interpreted as a free vehicle; it may represent unavailable or unreported MSRP information.


In [6]:
df["MSRP_Status"] = np.where(
    df["Base_MSRP"].eq(0),
    "Not Available / Zero",
    "Reported"
)

df["MSRP_Category"] = pd.cut(
    df["Base_MSRP"],
    bins=[-1, 25000, 50000, 75000, 100000, np.inf],
    labels=[
        "Under 25K",
        "25K–50K",
        "50K–75K",
        "75K–100K",
        "100K+"
    ]
)

display(df[["Base_MSRP", "MSRP_Status", "MSRP_Category"]].head())


,Base_MSRP,MSRP_Status,MSRP_Category
0,0,Not Available / Zero,Under 25K
1,0,Not Available / Zero,Under 25K
2,0,Not Available / Zero,Under 25K
3,0,Not Available / Zero,Under 25K
4,0,Not Available / Zero,Under 25K


# 8. Model-Year Segmentation


In [7]:
df["Model_Year_Group"] = pd.cut(
    df["Model_Year"],
    bins=[-np.inf, 2015, 2019, 2022, 2024, np.inf],
    labels=[
        "≤2015",
        "2016–2019",
        "2020–2022",
        "2023–2024",
        "2025+"
    ]
)

display(df["Model_Year_Group"].value_counts(dropna=False).sort_index())


Model_Year_Group
≤2015        15602
2016–2019    39382
2020–2022    57777
2023–2024    37721
2025+            0
Name: count, dtype: int64

# 9. CAFV Analytical Flag


In [8]:
df["CAFV_Flag"] = np.where(
    df["CAFV_Eligibility"]
    .astype("string")
    .str.contains("eligible", case=False, na=False),
    "Eligible",
    "Other"
)

display(df["CAFV_Flag"].value_counts(dropna=False))


CAFV_Flag
Eligible    80784
Other       69698
Name: count, dtype: int64

# 10. Vehicle-Level Analytical Flags


In [9]:
df["High_Range_Flag"] = np.where(
    df["Electric_Range"] >= df["Electric_Range"].median(),
    "Above/Equal Median",
    "Below Median"
)

df["High_MSRP_Flag"] = np.where(
    df["Base_MSRP"] >= df["Base_MSRP"].median(),
    "Above/Equal Median",
    "Below Median"
)

df["Newer_Vehicle_Flag"] = np.where(
    df["Vehicle_Age"] <= df["Vehicle_Age"].median(),
    "Newer",
    "Older"
)

display(df[
    [
        "High_Range_Flag",
        "High_MSRP_Flag",
        "Newer_Vehicle_Flag"
    ]
].head())


,High_Range_Flag,High_MSRP_Flag,Newer_Vehicle_Flag
0,Above/Equal Median,Above/Equal Median,Older
1,Above/Equal Median,Above/Equal Median,Newer
2,Above/Equal Median,Above/Equal Median,Newer
3,Above/Equal Median,Above/Equal Median,Older
4,Above/Equal Median,Above/Equal Median,Older


# 11. Basic Business Measures


In [10]:
total_vehicles = len(df)
total_makes = df["Make"].nunique()
total_models = df["Model"].nunique()
total_states = df["State"].nunique()

print("Total vehicles:", total_vehicles)
print("Manufacturers:", total_makes)
print("Models:", total_models)
print("States:", total_states)


Total vehicles: 150482
Manufacturers: 37
Models: 127
States: 41


# 12. Manufacturer Analytics

Build a reusable manufacturer-level analytical table.


In [11]:
manufacturer_analytics = (
    df.groupby("Make", dropna=False)
      .agg(
          Vehicle_Count=("VIN", "count"),
          Model_Count=("Model", "nunique"),
          State_Count=("State", "nunique"),
          Avg_Range=("Electric_Range", "mean"),
          Median_Range=("Electric_Range", "median"),
          Avg_MSRP=("Base_MSRP", "mean"),
          Median_MSRP=("Base_MSRP", "median"),
          Avg_Vehicle_Age=("Vehicle_Age", "mean"),
          Median_Vehicle_Age=("Vehicle_Age", "median"),
          CAFV_Eligible_Count=("CAFV_Flag", lambda x: (x == "Eligible").sum())
      )
      .reset_index()
)

manufacturer_analytics["Market_Share_%"] = (
    manufacturer_analytics["Vehicle_Count"]
    / total_vehicles * 100
)

manufacturer_analytics["CAFV_Rate_%"] = (
    manufacturer_analytics["CAFV_Eligible_Count"]
    / manufacturer_analytics["Vehicle_Count"] * 100
)

manufacturer_analytics["Volume_Rank"] = (
    manufacturer_analytics["Vehicle_Count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

manufacturer_analytics["Range_Rank"] = (
    manufacturer_analytics["Median_Range"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

display(
    manufacturer_analytics
    .sort_values("Vehicle_Count", ascending=False)
    .head(20)
)


,Make,Vehicle_Count,Model_Count,State_Count,Avg_Range,Median_Range,Avg_MSRP,Median_MSRP,Avg_Vehicle_Age,Median_Vehicle_Age,CAFV_Eligible_Count,Market_Share_%,CAFV_Rate_%,Volume_Rank,Range_Rank
31,TESLA,68983,5,35,90.30,0.00,"1,648.60",0.00,3.36,3.00,25898,45.84,37.54,1,22
25,NISSAN,13497,2,13,84.72,84.00,0.00,0.00,7.22,8.00,11003,8.97,81.52,2,4
6,CHEVROLET,12026,5,14,95.27,53.00,0.00,0.00,5.81,6.00,8849,7.99,73.58,3,7
10,FORD,7614,8,12,13.02,19.00,0.00,0.00,5.10,3.00,3958,5.06,51.98,4,18
4,BMW,6439,11,11,38.16,30.00,"4,154.12",0.00,4.30,3.00,5200,4.28,80.76,5,13
16,KIA,6198,7,9,52.62,26.00,"3,253.04",0.00,2.98,2.00,3545,4.12,57.20,6,14
33,TOYOTA,5223,6,12,26.92,25.00,0.00,0.00,5.45,5.00,5094,3.47,97.53,7,15
34,VOLKSWAGEN,4074,2,8,28.33,0.00,0.00,0.00,3.11,3.00,1075,2.71,26.39,8,22
35,VOLVO,3536,7,7,15.63,18.00,"4,785.93",0.00,2.73,2.00,2355,2.35,66.60,9,19
15,JEEP,3292,2,8,22.33,21.00,0.00,0.00,1.60,1.00,3292,2.19,100.00,10,17


# 13. Manufacturer Portfolio Diversity

Measure the breadth of each manufacturer's portfolio.


In [12]:
manufacturer_analytics["Portfolio_Diversity"] = (
    manufacturer_analytics["Model_Count"]
    / manufacturer_analytics["Model_Count"].max()
    * 100
)

manufacturer_analytics["Geographic_Presence_%"] = (
    manufacturer_analytics["State_Count"]
    / total_states * 100
)

display(
    manufacturer_analytics[
        [
            "Make",
            "Model_Count",
            "State_Count",
            "Portfolio_Diversity",
            "Geographic_Presence_%"
        ]
    ].sort_values("Model_Count", ascending=False).head(20)
)


,Make,Model_Count,State_Count,Portfolio_Diversity,Geographic_Presence_%
1,AUDI,11,6,100.00,14.63
4,BMW,11,11,100.00,26.83
22,MERCEDES-BENZ,10,2,90.91,4.88
10,FORD,8,12,72.73,29.27
13,HYUNDAI,8,6,72.73,14.63
35,VOLVO,7,7,63.64,17.07
16,KIA,7,9,63.64,21.95
33,TOYOTA,6,12,54.55,29.27
31,TESLA,5,35,45.45,85.37
6,CHEVROLET,5,14,45.45,34.15


# 14. Manufacturer Composite Analytical Score

This is a **project-defined analytical score**, not an official industry ranking.

The score combines normalized volume, range, portfolio diversity and geographic presence.


In [13]:
def minmax_score(series):
    s = series.astype(float)
    if s.max() == s.min():
        return pd.Series(100.0, index=s.index)
    return (s - s.min()) / (s.max() - s.min()) * 100

manufacturer_analytics["Volume_Score"] = minmax_score(
    manufacturer_analytics["Vehicle_Count"]
)

manufacturer_analytics["Range_Score"] = minmax_score(
    manufacturer_analytics["Median_Range"]
)

manufacturer_analytics["Portfolio_Score"] = minmax_score(
    manufacturer_analytics["Model_Count"]
)

manufacturer_analytics["Geographic_Score"] = minmax_score(
    manufacturer_analytics["State_Count"]
)

manufacturer_analytics["Overall_Analytical_Score"] = (
    0.40 * manufacturer_analytics["Volume_Score"]
    + 0.25 * manufacturer_analytics["Range_Score"]
    + 0.20 * manufacturer_analytics["Portfolio_Score"]
    + 0.15 * manufacturer_analytics["Geographic_Score"]
)

manufacturer_analytics["Overall_Rank"] = (
    manufacturer_analytics["Overall_Analytical_Score"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

display(
    manufacturer_analytics[
        [
            "Make",
            "Overall_Analytical_Score",
            "Overall_Rank"
        ]
    ].sort_values("Overall_Rank").head(20)
)


,Make,Overall_Analytical_Score,Overall_Rank
31,TESLA,63.00,1
4,BMW,31.35,2
6,CHEVROLET,26.37,3
1,AUDI,25.66,4
10,FORD,25.30,5
14,JAGUAR,25.13,6
25,NISSAN,24.09,7
16,KIA,21.90,8
33,TOYOTA,20.55,9
22,MERCEDES-BENZ,19.05,10


# 15. Model Analytics


In [14]:
model_analytics = (
    df.groupby(["Make", "Model"], dropna=False)
      .agg(
          Vehicle_Count=("VIN", "count"),
          State_Count=("State", "nunique"),
          Avg_Range=("Electric_Range", "mean"),
          Median_Range=("Electric_Range", "median"),
          Avg_MSRP=("Base_MSRP", "mean"),
          Median_MSRP=("Base_MSRP", "median"),
          Avg_Vehicle_Age=("Vehicle_Age", "mean"),
          Median_Vehicle_Age=("Vehicle_Age", "median")
      )
      .reset_index()
)

model_analytics["Market_Share_%"] = (
    model_analytics["Vehicle_Count"]
    / total_vehicles * 100
)

model_analytics["Volume_Rank"] = (
    model_analytics["Vehicle_Count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

display(
    model_analytics
    .sort_values("Vehicle_Count", ascending=False)
    .head(25)
)


,Make,Model,Vehicle_Count,State_Count,Avg_Range,Median_Range,Avg_MSRP,Median_MSRP,Avg_Vehicle_Age,Median_Vehicle_Age,Market_Share_%,Volume_Rank
108,TESLA,MODEL Y,28502,21,23.26,0.00,0.00,0.00,1.97,2.00,18.94,1
105,TESLA,MODEL 3,27709,29,121.38,215.00,0.00,0.00,3.59,4.00,18.41,2
91,NISSAN,LEAF,13187,13,86.72,84.00,0.00,0.00,7.37,8.00,8.76,3
106,TESLA,MODEL S,7611,12,183.46,208.00,"14,288.86",0.00,6.89,7.00,5.06,4
30,CHEVROLET,BOLT EV,5733,6,157.58,238.00,0.00,0.00,4.23,4.00,3.81,5
107,TESLA,MODEL X,5114,11,155.54,200.00,0.00,0.00,4.51,5.00,3.40,6
33,CHEVROLET,VOLT,4890,10,45.36,53.00,0.00,0.00,8.64,8.00,3.25,7
118,VOLKSWAGEN,ID.4,2999,8,0.00,0.00,0.00,0.00,1.89,2.00,1.99,8
62,KIA,NIRO,2876,6,80.40,26.00,0.00,0.00,3.13,3.00,1.91,9
34,CHRYSLER,PACIFICA,2642,11,32.24,32.00,"1,771.16",0.00,3.28,3.00,1.76,10


# 16. State / Geographic Analytics


In [15]:
state_analytics = (
    df.groupby("State", dropna=False)
      .agg(
          Vehicle_Count=("VIN", "count"),
          Manufacturer_Count=("Make", "nunique"),
          Model_Count=("Model", "nunique"),
          Avg_Range=("Electric_Range", "mean"),
          Median_Range=("Electric_Range", "median"),
          Avg_MSRP=("Base_MSRP", "mean"),
          Median_MSRP=("Base_MSRP", "median"),
          Avg_Vehicle_Age=("Vehicle_Age", "mean"),
          CAFV_Eligible_Count=("CAFV_Flag", lambda x: (x == "Eligible").sum())
      )
      .reset_index()
)

state_analytics["Market_Share_%"] = (
    state_analytics["Vehicle_Count"]
    / total_vehicles * 100
)

state_analytics["CAFV_Rate_%"] = (
    state_analytics["CAFV_Eligible_Count"]
    / state_analytics["Vehicle_Count"] * 100
)

state_analytics["Volume_Rank"] = (
    state_analytics["Vehicle_Count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

display(
    state_analytics
    .sort_values("Vehicle_Count", ascending=False)
    .head(20)
)


,State,Vehicle_Count,Manufacturer_Count,Model_Count,Avg_Range,Median_Range,Avg_MSRP,Median_MSRP,Avg_Vehicle_Age,CAFV_Eligible_Count,Market_Share_%,CAFV_Rate_%,Volume_Rank
39,WA,150141,37,127,67.86,18.00,"1,311.28",0.00,3.99,80560,99.77,53.66,1
6,CA,92,16,28,69.20,22.00,"1,998.91",0.00,4.83,59,0.06,64.13,2
38,VA,35,10,18,73.11,32.00,0.00,0.00,5.14,26,0.02,74.29,3
21,MD,33,14,20,57.36,30.00,"3,090.91",0.00,4.33,22,0.02,66.67,4
36,TX,20,8,12,80.35,23.00,"1,749.75",0.00,4.80,13,0.01,65.00,5
26,NC,13,4,6,40.69,0.00,0.00,0.00,3.85,6,0.01,46.15,6
15,IL,12,6,7,94.25,30.50,"3,332.92",0.00,4.50,8,0.01,66.67,7
7,CO,11,6,10,90.73,32.00,0.00,0.00,4.00,7,0.01,63.64,8
4,AZ,11,8,10,36.82,22.00,0.00,0.00,4.18,7,0.01,63.64,8
11,FL,9,4,6,123.44,42.00,0.00,0.00,5.00,7,0.01,77.78,9


# 17. State Diversity Score

Measure manufacturer and model diversity within each state.


In [16]:
state_analytics["Manufacturer_Diversity_Score"] = minmax_score(
    state_analytics["Manufacturer_Count"]
)

state_analytics["Model_Diversity_Score"] = minmax_score(
    state_analytics["Model_Count"]
)

state_analytics["State_Analytical_Score"] = (
    0.50 * minmax_score(state_analytics["Vehicle_Count"])
    + 0.25 * state_analytics["Manufacturer_Diversity_Score"]
    + 0.25 * state_analytics["Model_Diversity_Score"]
)

state_analytics["State_Analytical_Rank"] = (
    state_analytics["State_Analytical_Score"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

display(
    state_analytics[
        [
            "State",
            "Vehicle_Count",
            "Manufacturer_Count",
            "Model_Count",
            "State_Analytical_Score",
            "State_Analytical_Rank"
        ]
    ].sort_values("State_Analytical_Rank").head(20)
)


,State,Vehicle_Count,Manufacturer_Count,Model_Count,State_Analytical_Score,State_Analytical_Rank
39,WA,150141,37,127,100.00,1
6,CA,92,16,28,15.80,2
21,MD,33,14,20,12.81,3
38,VA,35,10,18,9.63,4
36,TX,20,8,12,7.05,5
4,AZ,11,8,10,6.65,6
7,CO,11,6,10,5.26,7
33,OR,8,6,8,4.86,8
15,IL,12,6,7,4.67,9
8,CT,7,6,6,4.47,10


# 18. Yearly Analytics


In [17]:
yearly_analytics = (
    df.groupby("Model_Year", dropna=False)
      .agg(
          Vehicle_Count=("VIN", "count"),
          Manufacturer_Count=("Make", "nunique"),
          Model_Count=("Model", "nunique"),
          Avg_Range=("Electric_Range", "mean"),
          Median_Range=("Electric_Range", "median"),
          Avg_MSRP=("Base_MSRP", "mean"),
          Median_MSRP=("Base_MSRP", "median"),
          Avg_Vehicle_Age=("Vehicle_Age", "mean")
      )
      .reset_index()
      .sort_values("Model_Year")
)

yearly_analytics["YoY_Growth_%"] = (
    yearly_analytics["Vehicle_Count"].pct_change() * 100
)

display(yearly_analytics)


,Model_Year,Vehicle_Count,Manufacturer_Count,Model_Count,Avg_Range,Median_Range,Avg_MSRP,Median_MSRP,Avg_Vehicle_Age,YoY_Growth_%
0,1997,1,1,1,39.00,39.00,0.00,0.00,27.00,NaN
1,1998,1,1,1,58.00,58.00,0.00,0.00,26.00,0.00
2,1999,4,1,1,74.00,74.00,0.00,0.00,25.00,300.00
3,2000,8,1,1,58.00,58.00,0.00,0.00,24.00,100.00
4,2002,2,1,1,95.00,95.00,0.00,0.00,22.00,-75.00
5,2003,1,1,1,95.00,95.00,0.00,0.00,21.00,-50.00
6,2008,19,1,1,220.00,220.00,"98,950.00","98,950.00",16.00,"1,800.00"
7,2010,24,2,2,226.88,245.00,"101,205.62","110,950.00",14.00,26.32
8,2011,796,5,5,70.85,73.00,958.54,0.00,13.00,"3,216.67"
9,2012,1633,8,9,62.19,35.00,"5,977.10",0.00,12.00,105.15


# 19. EV Type Analytics


In [18]:
ev_type_analytics = (
    df.groupby("EV_Type", dropna=False)
      .agg(
          Vehicle_Count=("VIN", "count"),
          Manufacturer_Count=("Make", "nunique"),
          Model_Count=("Model", "nunique"),
          Avg_Range=("Electric_Range", "mean"),
          Median_Range=("Electric_Range", "median"),
          Avg_MSRP=("Base_MSRP", "mean"),
          Median_MSRP=("Base_MSRP", "median"),
          Avg_Vehicle_Age=("Vehicle_Age", "mean")
      )
      .reset_index()
)

ev_type_analytics["Market_Share_%"] = (
    ev_type_analytics["Vehicle_Count"]
    / total_vehicles * 100
)

display(ev_type_analytics)


,EV_Type,Vehicle_Count,Manufacturer_Count,Model_Count,Avg_Range,Median_Range,Avg_MSRP,Median_MSRP,Avg_Vehicle_Age,Market_Share_%
0,Battery Electric Vehicle (BEV),116807,27,67,78.61,0.00,"1,146.23",0.00,3.69,77.62
1,Plug-in Hybrid Electric Vehicle (PHEV),33675,25,63,30.66,26.00,"1,889.88",0.00,5.06,22.38


# 20. Range Segment Analytics


In [19]:
range_analytics = (
    df.groupby("Range_Category", observed=True)
      .agg(
          Vehicle_Count=("VIN", "count"),
          Manufacturer_Count=("Make", "nunique"),
          Model_Count=("Model", "nunique"),
          Avg_MSRP=("Base_MSRP", "mean"),
          Avg_Vehicle_Age=("Vehicle_Age", "mean")
      )
      .reset_index()
)

range_analytics["Share_%"] = (
    range_analytics["Vehicle_Count"]
    / total_vehicles * 100
)

display(range_analytics)


,Range_Category,Vehicle_Count,Manufacturer_Count,Model_Count,Avg_MSRP,Avg_Vehicle_Age,Share_%
0,0–100,113200,37,125,718.12,3.43,75.22
1,101–200,6613,9,9,379.90,5.94,4.39
2,201–300,28010,9,14,"4,060.17",5.81,18.61
3,301–400,2659,1,2,0.00,4.00,1.77


# 21. MSRP Segment Analytics


In [20]:
msrp_analytics = (
    df.groupby("MSRP_Category", observed=True)
      .agg(
          Vehicle_Count=("VIN", "count"),
          Manufacturer_Count=("Make", "nunique"),
          Model_Count=("Model", "nunique"),
          Avg_Range=("Electric_Range", "mean"),
          Avg_Vehicle_Age=("Vehicle_Age", "mean")
      )
      .reset_index()
)

msrp_analytics["Share_%"] = (
    msrp_analytics["Vehicle_Count"]
    / total_vehicles * 100
)

display(msrp_analytics)


,MSRP_Category,Vehicle_Count,Manufacturer_Count,Model_Count,Avg_Range,Avg_Vehicle_Age,Share_%
0,Under 25K,147027,35,120,66.57,3.89,97.70
1,25K–50K,1154,6,7,59.89,6.70,0.77
2,50K–75K,2161,3,4,159.54,9.32,1.44
3,75K–100K,83,4,4,64.23,7.88,0.06
4,100K+,57,3,4,133.11,11.65,0.04


# 22. Age Segment Analytics


In [21]:
age_analytics = (
    df.groupby("Age_Group", observed=True)
      .agg(
          Vehicle_Count=("VIN", "count"),
          Manufacturer_Count=("Make", "nunique"),
          Model_Count=("Model", "nunique"),
          Avg_Range=("Electric_Range", "mean"),
          Avg_MSRP=("Base_MSRP", "mean")
      )
      .reset_index()
)

age_analytics["Share_%"] = (
    age_analytics["Vehicle_Count"]
    / total_vehicles * 100
)

display(age_analytics)


,Age_Group,Vehicle_Count,Manufacturer_Count,Model_Count,Avg_Range,Avg_MSRP,Share_%
0,0–2 Years,65520,30,82,4.60,0.00,43.54
1,3–5 Years,40695,26,60,118.25,536.82,27.04
2,6–10 Years,37212,20,50,122.68,"2,868.65",24.73
3,10+ Years,7055,12,18,75.90,"9,771.11",4.69


# 23. Manufacturer × EV Type


In [22]:
manufacturer_ev_type = (
    pd.crosstab(
        df["Make"],
        df["EV_Type"],
        margins=False
    )
    .reset_index()
)

display(manufacturer_ev_type.head(20))


EV_Type,Make,Battery Electric Vehicle (BEV),Plug-in Hybrid Electric Vehicle (PHEV)
0,ALFA ROMEO,0,12
1,AUDI,1662,1344
2,AZURE DYNAMICS,9,0
3,BENTLEY,0,2
4,BMW,1796,4643
5,CADILLAC,84,92
6,CHEVROLET,7136,4890
7,CHRYSLER,0,2642
8,FIAT,806,0
9,FISKER,0,17


# 24. State × EV Type


In [23]:
state_ev_type = (
    pd.crosstab(
        df["State"],
        df["EV_Type"],
        margins=False
    )
    .reset_index()
)

display(state_ev_type.head(20))


EV_Type,State,Battery Electric Vehicle (BEV),Plug-in Hybrid Electric Vehicle (PHEV)
0,AK,1,0
1,AL,3,0
2,AP,1,0
3,AR,2,0
4,AZ,6,5
5,BC,2,0
6,CA,61,31
7,CO,9,2
8,CT,1,6
9,DC,5,0


# 25. Manufacturer × Range Category


In [24]:
manufacturer_range = (
    pd.crosstab(
        df["Make"],
        df["Range_Category"],
        normalize="index"
    )
    .mul(100)
    .round(2)
    .reset_index()
)

display(manufacturer_range.head(20))


Range_Category,Make,0–100,101–200,201–300,301–400
0,ALFA ROMEO,100.00,0.00,0.00,0.00
1,AUDI,78.38,0.00,21.62,0.00
2,AZURE DYNAMICS,100.00,0.00,0.00,0.00
3,BENTLEY,100.00,0.00,0.00,0.00
4,BMW,94.28,5.72,0.00,0.00
5,CADILLAC,100.00,0.00,0.00,0.00
6,CHEVROLET,69.16,0.00,30.84,0.00
7,CHRYSLER,100.00,0.00,0.00,0.00
8,FIAT,100.00,0.00,0.00,0.00
9,FISKER,100.00,0.00,0.00,0.00


# 26. State × Age Group


In [25]:
state_age = (
    pd.crosstab(
        df["State"],
        df["Age_Group"],
        normalize="index"
    )
    .mul(100)
    .round(2)
    .reset_index()
)

display(state_age.head(20))


Age_Group,State,0–2 Years,3–5 Years,6–10 Years,10+ Years
0,AK,0.00,100.00,0.00,0.00
1,AL,66.67,33.33,0.00,0.00
2,AP,0.00,0.00,0.00,100.00
3,AR,0.00,100.00,0.00,0.00
4,AZ,27.27,45.45,18.18,9.09
5,BC,0.00,50.00,0.00,50.00
6,CA,28.26,32.61,33.70,5.43
7,CO,36.36,36.36,27.27,0.00
8,CT,14.29,28.57,57.14,0.00
9,DC,40.00,40.00,20.00,0.00


# 27. Manufacturer Market Share


In [26]:
market_share_analytics = (
    manufacturer_analytics[
        ["Make", "Vehicle_Count", "Market_Share_%"]
    ]
    .sort_values("Vehicle_Count", ascending=False)
    .reset_index(drop=True)
)

market_share_analytics["Cumulative_Share_%"] = (
    market_share_analytics["Market_Share_%"].cumsum()
)

display(market_share_analytics.head(20))


,Make,Vehicle_Count,Market_Share_%,Cumulative_Share_%
0,TESLA,68983,45.84,45.84
1,NISSAN,13497,8.97,54.81
2,CHEVROLET,12026,7.99,62.80
3,FORD,7614,5.06,67.86
4,BMW,6439,4.28,72.14
5,KIA,6198,4.12,76.26
6,TOYOTA,5223,3.47,79.73
7,VOLKSWAGEN,4074,2.71,82.44
8,VOLVO,3536,2.35,84.79
9,JEEP,3292,2.19,86.98


# 28. Percentile Analysis

Percentiles help identify high-performing manufacturers/models without relying only on averages.


In [27]:
df["Range_Percentile"] = (
    df["Electric_Range"].rank(pct=True) * 100
)

df["MSRP_Percentile"] = (
    df["Base_MSRP"].rank(pct=True) * 100
)

df["Age_Percentile"] = (
    df["Vehicle_Age"].rank(pct=True) * 100
)

display(df[
    [
        "Electric_Range",
        "Range_Percentile",
        "Base_MSRP",
        "MSRP_Percentile",
        "Vehicle_Age",
        "Age_Percentile"
    ]
].head())


,Electric_Range,Range_Percentile,Base_MSRP,MSRP_Percentile,Vehicle_Age,Age_Percentile
0,258,94.01,0,48.85,4,59.71
1,25,55.98,0,48.85,2,34.30
2,25,55.98,0,48.85,1,12.75
3,215,85.10,0,48.85,6,75.38
4,97,75.06,0,48.85,6,75.38


# 29. Vehicle Performance Segment

Create a simple analytical segmentation using range and age.

This is a project-specific segmentation, not a market-standard classification.


In [28]:
range_median = df["Electric_Range"].median()
age_median = df["Vehicle_Age"].median()

df["Vehicle_Performance_Segment"] = np.select(
    [
        (df["Electric_Range"] >= range_median) &
        (df["Vehicle_Age"] <= age_median),

        (df["Electric_Range"] >= range_median) &
        (df["Vehicle_Age"] > age_median),

        (df["Electric_Range"] < range_median) &
        (df["Vehicle_Age"] <= age_median)
    ],
    [
        "High Range / Newer",
        "High Range / Older",
        "Lower Range / Newer"
    ],
    default="Lower Range / Older"
)

display(df["Vehicle_Performance_Segment"].value_counts())


Vehicle_Performance_Segment
Lower Range / Newer    69992
High Range / Older     62138
High Range / Newer     14212
Lower Range / Older     4140
Name: count, dtype: int64

# 30. Manufacturer Range vs Volume Segmentation


In [29]:
volume_median = manufacturer_analytics["Vehicle_Count"].median()
range_median_make = manufacturer_analytics["Median_Range"].median()

manufacturer_analytics["Market_Segment"] = np.select(
    [
        (manufacturer_analytics["Vehicle_Count"] >= volume_median) &
        (manufacturer_analytics["Median_Range"] >= range_median_make),

        (manufacturer_analytics["Vehicle_Count"] >= volume_median) &
        (manufacturer_analytics["Median_Range"] < range_median_make),

        (manufacturer_analytics["Vehicle_Count"] < volume_median) &
        (manufacturer_analytics["Median_Range"] >= range_median_make)
    ],
    [
        "High Volume / High Range",
        "High Volume / Lower Range",
        "Lower Volume / High Range"
    ],
    default="Lower Volume / Lower Range"
)

display(
    manufacturer_analytics["Market_Segment"]
    .value_counts()
)


Market_Segment
Lower Volume / High Range     10
High Volume / Lower Range     10
High Volume / High Range       9
Lower Volume / Lower Range     8
Name: count, dtype: int64

# 31. Portfolio Concentration

Measure how concentrated the vehicle population is among manufacturers.


In [30]:
top_5_share = (
    market_share_analytics
    .head(5)["Market_Share_%"]
    .sum()
)

top_10_share = (
    market_share_analytics
    .head(10)["Market_Share_%"]
    .sum()
)

print(f"Top 5 manufacturer share: {top_5_share:.2f}%")
print(f"Top 10 manufacturer share: {top_10_share:.2f}%")


Top 5 manufacturer share: 72.14%
Top 10 manufacturer share: 86.98%


# 32. Model Concentration


In [31]:
model_market_share = (
    model_analytics[
        ["Make", "Model", "Vehicle_Count"]
    ]
    .sort_values("Vehicle_Count", ascending=False)
    .reset_index(drop=True)
)

model_market_share["Market_Share_%"] = (
    model_market_share["Vehicle_Count"]
    / total_vehicles * 100
)

model_market_share["Cumulative_Share_%"] = (
    model_market_share["Market_Share_%"].cumsum()
)

display(model_market_share.head(20))


,Make,Model,Vehicle_Count,Market_Share_%,Cumulative_Share_%
0,TESLA,MODEL Y,28502,18.94,18.94
1,TESLA,MODEL 3,27709,18.41,37.35
2,NISSAN,LEAF,13187,8.76,46.12
3,TESLA,MODEL S,7611,5.06,51.17
4,CHEVROLET,BOLT EV,5733,3.81,54.98
5,TESLA,MODEL X,5114,3.40,58.38
6,CHEVROLET,VOLT,4890,3.25,61.63
7,VOLKSWAGEN,ID.4,2999,1.99,63.63
8,KIA,NIRO,2876,1.91,65.54
9,CHRYSLER,PACIFICA,2642,1.76,67.29


# 33. Manufacturer Portfolio Matrix

Combine model breadth and geographic breadth.


In [32]:
portfolio_matrix = manufacturer_analytics[
    [
        "Make",
        "Model_Count",
        "State_Count",
        "Vehicle_Count",
        "Median_Range",
        "Median_MSRP"
    ]
].copy()

portfolio_matrix["Portfolio_Breadth_Score"] = (
    0.5 * minmax_score(portfolio_matrix["Model_Count"])
    + 0.5 * minmax_score(portfolio_matrix["State_Count"])
)

display(
    portfolio_matrix
    .sort_values("Portfolio_Breadth_Score", ascending=False)
    .head(20)
)


,Make,Model_Count,State_Count,Vehicle_Count,Median_Range,Median_MSRP,Portfolio_Breadth_Score
31,TESLA,5,35,68983,0.00,0.00,70.00
4,BMW,11,11,6439,30.00,0.00,64.71
1,AUDI,11,6,3006,16.00,0.00,57.35
10,FORD,8,12,7614,19.00,0.00,51.18
22,MERCEDES-BENZ,10,2,1054,0.00,0.00,46.47
13,HYUNDAI,8,6,3171,0.00,0.00,42.35
16,KIA,7,9,6198,26.00,0.00,41.76
33,TOYOTA,6,12,5223,25.00,0.00,41.18
6,CHEVROLET,5,14,12026,53.00,0.00,39.12
35,VOLVO,7,7,3536,18.00,0.00,38.82


# 34. Model Range-to-Price Indicator

For vehicles with a reported positive MSRP, calculate a simple range-per-price indicator.

This should be interpreted cautiously because electric range and MSRP are not sufficient to measure vehicle value comprehensively.


In [33]:
df["Range_per_10K_MSRP"] = np.where(
    df["Base_MSRP"] > 0,
    df["Electric_Range"] / (df["Base_MSRP"] / 10000),
    np.nan
)

display(
    df[
        [
            "Make",
            "Model",
            "Electric_Range",
            "Base_MSRP",
            "Range_per_10K_MSRP"
        ]
    ].head(20)
)


,Make,Model,Electric_Range,Base_MSRP,Range_per_10K_MSRP
0,HYUNDAI,KONA,258,0,NaN
1,JEEP,GRAND CHEROKEE,25,0,NaN
2,JEEP,GRAND CHEROKEE,25,0,NaN
3,TESLA,MODEL 3,215,0,NaN
4,BMW,I3,97,0,NaN
5,TESLA,MODEL 3,266,0,NaN
6,CHRYSLER,PACIFICA,33,0,NaN
7,TESLA,MODEL Y,291,0,NaN
8,TESLA,MODEL 3,215,0,NaN
9,TESLA,MODEL Y,0,0,NaN


# 35. Manufacturer Range-to-Price Summary


In [34]:
range_price_analytics = (
    df[df["Base_MSRP"] > 0]
    .groupby("Make")
    .agg(
        Median_Range_per_10K_MSRP=(
            "Range_per_10K_MSRP", "median"
        ),
        Vehicle_Count=("VIN", "count")
    )
    .reset_index()
)

display(
    range_price_analytics
    .sort_values(
        "Median_Range_per_10K_MSRP",
        ascending=False
    )
    .head(20)
)


,Make,Median_Range_per_10K_MSRP,Vehicle_Count
10,WHEEGO ELECTRIC CARS,30.31,3
8,TESLA,29.76,1622
4,KIA,29.11,625
2,CHRYSLER,8.00,117
7,SUBARU,4.86,64
1,CADILLAC,4.13,15
5,MINI,3.25,154
3,FISKER,3.24,17
9,VOLVO,3.21,301
0,BMW,3.04,506


# 36. City-Level Analytics

Use city as a finer geographic dimension.


In [35]:
city_analytics = (
    df.groupby(["State", "City"], dropna=False)
      .agg(
          Vehicle_Count=("VIN", "count"),
          Manufacturer_Count=("Make", "nunique"),
          Model_Count=("Model", "nunique"),
          Avg_Range=("Electric_Range", "mean"),
          Median_Range=("Electric_Range", "median"),
          Avg_MSRP=("Base_MSRP", "mean")
      )
      .reset_index()
)

city_analytics["State_Share_%"] = (
    city_analytics["Vehicle_Count"]
    / city_analytics.groupby("State")["Vehicle_Count"].transform("sum")
    * 100
)

display(
    city_analytics
    .sort_values("Vehicle_Count", ascending=False)
    .head(25)
)


,State,City,Vehicle_Count,Manufacturer_Count,Model_Count,Avg_Range,Median_Range,Avg_MSRP,State_Share_%
588,WA,Seattle,25675,34,118,71.27,19.00,"1,231.07",17.10
258,WA,Bellevue,7690,32,108,72.09,0.00,"1,551.07",5.12
562,WA,Redmond,5501,32,100,70.69,0.00,"1,224.31",3.66
660,WA,Vancouver,5310,31,101,64.25,21.00,"1,171.61",3.54
265,WA,Bothell,4861,31,99,61.19,0.00,696.25,3.24
421,WA,Kirkland,4622,31,103,76.48,6.00,"1,574.33",3.08
584,WA,Sammamish,4436,32,99,72.61,0.00,"1,473.15",2.95
563,WA,Renton,4043,30,97,60.17,0.00,"1,085.25",2.69
516,WA,Olympia,3634,31,93,70.79,25.00,"1,015.53",2.42
632,WA,Tacoma,3121,30,97,63.94,20.00,"1,319.27",2.08


# 37. County-Level Analytics


In [36]:
county_analytics = (
    df.groupby(["State", "County"], dropna=False)
      .agg(
          Vehicle_Count=("VIN", "count"),
          Manufacturer_Count=("Make", "nunique"),
          Model_Count=("Model", "nunique"),
          Avg_Range=("Electric_Range", "mean"),
          Median_Range=("Electric_Range", "median")
      )
      .reset_index()
)

display(
    county_analytics
    .sort_values("Vehicle_Count", ascending=False)
    .head(25)
)


,State,County,Vehicle_Count,Manufacturer_Count,Model_Count,Avg_Range,Median_Range
172,WA,King,79075,35,123,68.56,14.00
186,WA,Snohomish,17307,34,117,63.13,0.00
182,WA,Pierce,11542,33,116,64.38,19.00
161,WA,Clark,8849,33,110,65.57,21.00
189,WA,Thurston,5403,32,100,69.34,22.00
173,WA,Kitsap,4923,33,104,68.94,25.00
187,WA,Spokane,3690,31,102,63.92,21.00
192,WA,Whatcom,3668,34,103,71.41,26.50
158,WA,Benton,1800,28,86,70.74,25.00
184,WA,Skagit,1658,31,84,73.57,25.00


# 38. Utility-Level Analytics

Electric utility can be used as an infrastructure/market proxy where the source definitions support that interpretation.


In [37]:
utility_analytics = (
    df.groupby("Electric_Utility", dropna=False)
      .agg(
          Vehicle_Count=("VIN", "count"),
          State_Count=("State", "nunique"),
          Manufacturer_Count=("Make", "nunique"),
          Model_Count=("Model", "nunique")
      )
      .reset_index()
      .sort_values("Vehicle_Count", ascending=False)
)

display(utility_analytics.head(25))


,Electric_Utility,Vehicle_Count,State_Count,Manufacturer_Count,Model_Count
74,PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),55637,3,35,122
73,PUGET SOUND ENERGY INC,29865,1,36,121
57,CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),27268,1,34,118
36,BONNEVILLE POWER ADMINISTRATION||PUD NO 1 OF C...,8643,1,33,109
19,BONNEVILLE POWER ADMINISTRATION||CITY OF TACOM...,6622,1,33,111
75,PUGET SOUND ENERGY INC||PUD NO 1 OF WHATCOM CO...,3443,1,33,102
2,BONNEVILLE POWER ADMINISTRATION||AVISTA CORP||...,2244,1,30,93
31,BONNEVILLE POWER ADMINISTRATION||PUD 1 OF SNOH...,1341,1,30,84
65,PACIFICORP,1102,1,31,86
34,BONNEVILLE POWER ADMINISTRATION||PUD NO 1 OF B...,1048,1,28,81


# 39. Vehicle Location Coverage


In [38]:
location_coverage = (
    df["Vehicle_Location"]
    .value_counts(dropna=False)
    .rename_axis("Vehicle_Location")
    .reset_index(name="Vehicle_Count")
)

location_coverage["Share_%"] = (
    location_coverage["Vehicle_Count"]
    / total_vehicles * 100
)

display(location_coverage.head(20))


,Vehicle_Location,Vehicle_Count,Share_%
0,POINT (-122.12302 47.67668),3876,2.58
1,POINT (-122.1876761 47.820517),2753,1.83
2,POINT (-122.20264 47.6785),2619,1.74
3,POINT (-122.16937 47.571015),2457,1.63
4,POINT (-122.201905 47.61385),2456,1.63
5,POINT (-122.3185 47.67949),2359,1.57
6,POINT (-122.0313266 47.6285782),2141,1.42
7,POINT (-122.151665 47.75855),2095,1.39
8,POINT (-122.2377542 47.582905),2083,1.38
9,POINT (-122.29179 47.43473),2074,1.38


# 40. Model-Year × EV Type Analytics


In [39]:
year_ev_type = (
    df.groupby(
        ["Model_Year", "EV_Type"],
        dropna=False
    )
    .size()
    .reset_index(name="Vehicle_Count")
)

year_ev_type["Year_Share_%"] = (
    year_ev_type["Vehicle_Count"]
    / year_ev_type.groupby("Model_Year")["Vehicle_Count"].transform("sum")
    * 100
)

display(year_ev_type.head(30))


,Model_Year,EV_Type,Vehicle_Count,Year_Share_%
0,1997,Battery Electric Vehicle (BEV),1,100.00
1,1998,Battery Electric Vehicle (BEV),1,100.00
2,1999,Battery Electric Vehicle (BEV),4,100.00
3,2000,Battery Electric Vehicle (BEV),8,100.00
4,2002,Battery Electric Vehicle (BEV),2,100.00
5,2003,Battery Electric Vehicle (BEV),1,100.00
6,2008,Battery Electric Vehicle (BEV),19,100.00
7,2010,Battery Electric Vehicle (BEV),21,87.50
8,2010,Plug-in Hybrid Electric Vehicle (PHEV),3,12.50
9,2011,Battery Electric Vehicle (BEV),718,90.20


# 41. Make × Model Portfolio Table


In [40]:
make_model_portfolio = (
    df.groupby(["Make", "Model"], dropna=False)
      .agg(
          Vehicle_Count=("VIN", "count"),
          Median_Range=("Electric_Range", "median"),
          Median_MSRP=("Base_MSRP", "median"),
          Median_Age=("Vehicle_Age", "median"),
          State_Count=("State", "nunique")
      )
      .reset_index()
)

make_model_portfolio["Model_Share_%"] = (
    make_model_portfolio["Vehicle_Count"]
    / total_vehicles * 100
)

display(
    make_model_portfolio
    .sort_values("Vehicle_Count", ascending=False)
    .head(30)
)


,Make,Model,Vehicle_Count,Median_Range,Median_MSRP,Median_Age,State_Count,Model_Share_%
108,TESLA,MODEL Y,28502,0.00,0.00,2.00,21,18.94
105,TESLA,MODEL 3,27709,215.00,0.00,4.00,29,18.41
91,NISSAN,LEAF,13187,84.00,0.00,8.00,13,8.76
106,TESLA,MODEL S,7611,208.00,0.00,7.00,12,5.06
30,CHEVROLET,BOLT EV,5733,238.00,0.00,4.00,6,3.81
107,TESLA,MODEL X,5114,200.00,0.00,5.00,11,3.40
33,CHEVROLET,VOLT,4890,53.00,0.00,8.00,10,3.25
118,VOLKSWAGEN,ID.4,2999,0.00,0.00,2.00,8,1.99
62,KIA,NIRO,2876,26.00,0.00,3.00,6,1.91
34,CHRYSLER,PACIFICA,2642,32.00,0.00,3.00,11,1.76


# 42. Executive KPI Table


In [41]:
executive_kpis = pd.DataFrame({
    "KPI": [
        "Total Vehicles",
        "Unique Manufacturers",
        "Unique Models",
        "States Covered",
        "EV Types",
        "Average Electric Range",
        "Median Electric Range",
        "Average MSRP",
        "Median MSRP",
        "Median Vehicle Age",
        "Top 5 Manufacturer Share %",
        "Top 10 Manufacturer Share %"
    ],
    "Value": [
        total_vehicles,
        total_makes,
        total_models,
        total_states,
        df["EV_Type"].nunique(),
        df["Electric_Range"].mean(),
        df["Electric_Range"].median(),
        df["Base_MSRP"].mean(),
        df["Base_MSRP"].median(),
        df["Vehicle_Age"].median(),
        top_5_share,
        top_10_share
    ]
})

display(executive_kpis)


,KPI,Value
0,Total Vehicles,"150,482.00"
1,Unique Manufacturers,37.00
2,Unique Models,127.00
3,States Covered,41.00
4,EV Types,2.00
5,Average Electric Range,67.88
6,Median Electric Range,18.00
7,Average MSRP,"1,312.64"
8,Median MSRP,0.00
9,Median Vehicle Age,3.00


# 43. ML-Ready Feature Dataset

Create a compact analytical dataset containing engineered numeric and categorical features.

Categorical encoding is intentionally left for the ML preprocessing notebook.


In [42]:
ml_feature_columns = [
    "Model_Year",
    "Vehicle_Age",
    "Electric_Range",
    "Base_MSRP",
    "Legislative District",
    "Make",
    "Model",
    "State",
    "EV_Type",
    "CAFV_Eligibility",
    "Age_Group",
    "Range_Category",
    "MSRP_Category",
    "Model_Year_Group",
    "CAFV_Flag",
    "High_Range_Flag",
    "High_MSRP_Flag",
    "Newer_Vehicle_Flag",
    "Vehicle_Performance_Segment",
    "Range_per_10K_MSRP"
]

ml_feature_columns = [
    col for col in ml_feature_columns
    if col in df.columns
]

ml_features = df[ml_feature_columns].copy()

display(ml_features.head())
print("ML feature shape:", ml_features.shape)


,Model_Year,Vehicle_Age,Electric_Range,Base_MSRP,Make,Model,State,EV_Type,CAFV_Eligibility,Age_Group,Range_Category,MSRP_Category,Model_Year_Group,CAFV_Flag,High_Range_Flag,High_MSRP_Flag,Newer_Vehicle_Flag,Vehicle_Performance_Segment,Range_per_10K_MSRP
0,2020,4,258,0,HYUNDAI,KONA,WA,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,3–5 Years,201–300,Under 25K,2020–2022,Eligible,Above/Equal Median,Above/Equal Median,Older,High Range / Older,NaN
1,2022,2,25,0,JEEP,GRAND CHEROKEE,WA,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,0–2 Years,0–100,Under 25K,2020–2022,Eligible,Above/Equal Median,Above/Equal Median,Newer,High Range / Newer,NaN
2,2023,1,25,0,JEEP,GRAND CHEROKEE,WA,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,0–2 Years,0–100,Under 25K,2023–2024,Eligible,Above/Equal Median,Above/Equal Median,Newer,High Range / Newer,NaN
3,2018,6,215,0,TESLA,MODEL 3,WA,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,6–10 Years,201–300,Under 25K,2016–2019,Eligible,Above/Equal Median,Above/Equal Median,Older,High Range / Older,NaN
4,2018,6,97,0,BMW,I3,WA,Plug-in Hybrid Electric Vehicle (PHEV),Clean Alternative Fuel Vehicle Eligible,6–10 Years,0–100,Under 25K,2016–2019,Eligible,Above/Equal Median,Above/Equal Median,Older,High Range / Older,NaN


ML feature shape: (150482, 19)


# 44. Dashboard Summary Tables

These tables can later be consumed by Power BI.


In [43]:
dashboard_tables = {
    "manufacturer": manufacturer_analytics,
    "model": model_analytics,
    "state": state_analytics,
    "year": yearly_analytics,
    "ev_type": ev_type_analytics,
    "range": range_analytics,
    "msrp": msrp_analytics,
    "age": age_analytics,
    "city": city_analytics,
    "county": county_analytics,
    "utility": utility_analytics
}

for name, table in dashboard_tables.items():
    print(f"{name:15} -> {table.shape}")


manufacturer    -> (37, 24)
model           -> (127, 12)
state           -> (41, 17)
year            -> (22, 10)
ev_type         -> (2, 10)
range           -> (4, 7)
msrp            -> (5, 7)
age             -> (4, 7)
city            -> (696, 9)
county          -> (197, 7)
utility         -> (76, 5)


# 45. Export Processed Analytical Data

The outputs are stored separately from the cleaned source data.


In [44]:
OUTPUT_DIR = Path("../datas/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

exports = {
    "ev_features.csv": df,
    "ml_features.csv": ml_features,
    "manufacturer_analytics.csv": manufacturer_analytics,
    "model_analytics.csv": model_analytics,
    "state_analytics.csv": state_analytics,
    "yearly_analytics.csv": yearly_analytics,
    "ev_type_analytics.csv": ev_type_analytics,
    "range_analytics.csv": range_analytics,
    "msrp_analytics.csv": msrp_analytics,
    "age_analytics.csv": age_analytics,
    "market_share_analytics.csv": market_share_analytics,
    "city_analytics.csv": city_analytics,
    "county_analytics.csv": county_analytics,
    "utility_analytics.csv": utility_analytics,
    "executive_kpis.csv": executive_kpis
}

for filename, table in exports.items():
    table.to_csv(OUTPUT_DIR / filename, index=False)

print("✅ Export completed")
print("Output folder:", OUTPUT_DIR.resolve())


✅ Export completed
Output folder: D:\ev_project_v2-Data Analysis\datas\processed


# 46. Final Data Validation

Confirm that the analytical outputs were created successfully.


In [45]:
print("Final vehicle-level shape:", df.shape)

print("\nProcessed files:")
for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print("✓", path.name)

print("\nFinal missing-value summary:")
display(
    df.isnull()
      .sum()
      .sort_values(ascending=False)
      .head(20)
)


Final vehicle-level shape: (150482, 32)

Processed files:
✓ age_analytics.csv
✓ city_analytics.csv
✓ county_analytics.csv
✓ ev_features.csv
✓ ev_statistical_profile.csv
✓ ev_type_analytics.csv
✓ executive_kpis.csv
✓ manufacturer_analytics.csv
✓ market_share_analytics.csv
✓ ml_features.csv
✓ model_analytics.csv
✓ msrp_analytics.csv
✓ range_analytics.csv
✓ state_analytics.csv
✓ utility_analytics.csv
✓ yearly_analytics.csv

Final missing-value summary:


Range_per_10K_MSRP      147027
VIN                          0
City                         0
County                       0
Postal_Code                  0
Model_Year                   0
Make                         0
State                        0
EV_Type                      0
CAFV_Eligibility             0
Electric_Range               0
Base_MSRP                    0
Vehicle_ID                   0
Vehicle_Location             0
Electric_Utility             0
Model                        0
Census_Tract                 0
Legislative_District         0
Age_Group                    0
Vehicle_Age                  0
dtype: int64

# 47. Business Insight Template

Do not write conclusions directly from a single metric.

Use this structure:

### Observation
What does the table/chart show?

### Evidence
Which metric supports the observation?

### Business Interpretation
Why could this matter to an automotive, mobility, energy or infrastructure business?

### Limitation
What does the dataset not prove?

### Recommended Action
What could a decision-maker investigate next?

Example:

> **Observation:** A small number of manufacturers account for a large share of the observed vehicle records.  
> **Evidence:** Top-5 cumulative manufacturer share.  
> **Interpretation:** The observed market is relatively concentrated within the dataset.  
> **Limitation:** Dataset coverage and registration methodology may affect representation.  
> **Action:** Investigate concentration by state, model year and EV type.


# 48. Notebook Conclusion

## What was created?

```text
Cleaned Dataset
      ↓
Vehicle Features
      ↓
Age / Range / MSRP Segmentation
      ↓
Manufacturer Analytics
      ↓
Model Analytics
      ↓
State / City / County Analytics
      ↓
EV Type Analytics
      ↓
Market Share
      ↓
Rankings
      ↓
Portfolio Scores
      ↓
Business Segments
      ↓
ML-Ready Features
      ↓
Processed CSV Layer
```

## Next step

The next notebook can consume these processed tables for **advanced visualization / dashboard preparation**, followed by the final Power BI report.

### Important analytical rule

Project-defined scores and segments are analytical constructs. They should be clearly labelled as such and should not be presented as official market ratings.
